# First-Time Setup — Why Is My LLM Slow?

**Run this notebook exactly once, before you start Chapter 0.**

It will:
1. Mount your Google Drive
2. Clone the book's repository into Drive
3. Install all required packages and verify your GPU
4. Download LLaMA-3.2-1B weights to Drive

After this notebook completes, close it. Every chapter notebook is now in
`MyDrive/llm-perf-book-labs/notebooks/` and the model weights are in
`MyDrive/llm-perf-book-labs/models/`. You will never need to re-run this notebook.

---

> **Before running:** Make sure you are on a T4 GPU runtime.
> **Runtime → Change runtime type → T4 GPU → Save**, then click **Connect**.

## Step 1 — Mount Google Drive

This makes your Drive accessible inside this VM at `/content/drive/MyDrive/`.
You will be asked to sign in and grant access — do this once, then the button
just requires a single click on future sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 — Clone the repository into Drive

This clones the entire book repository — all chapter notebooks and the setup
script — into `MyDrive/llm-perf-book-labs/`. The clone happens on Colab's fast network
connection, not yours. Takes about 10–20 seconds.

In [ ]:
import os

REPO_URL  = "https://github.com/Nitin-x2a/llm-gpu-perf.git"
DEST_DIR  = "/content/drive/MyDrive/llm-perf-book-labs"

if os.path.exists(os.path.join(DEST_DIR, ".git")):
    print(f"Repo already on Drive — pulling latest changes ...")
    ret = os.system(f"git -C {DEST_DIR} pull")
    if ret != 0:
        print("⚠  git pull failed — continuing with existing files.")
else:
    print(f"Cloning into {DEST_DIR} ...")
    ret = os.system(f"git clone {REPO_URL} {DEST_DIR}")
    if ret != 0:
        raise RuntimeError("git clone failed. Check the repo URL above.")

print("\nDone. Contents:")
os.system(f"ls {DEST_DIR}")

## Step 3 — Install packages and verify your GPU

Runs `env_setup.py` (now on your Drive from the clone above). Installs
PyTorch, Triton, bitsandbytes, transformers, and accelerate, then prints
a verification summary including your GPU. Takes ~2 minutes on a fresh VM.

In [ ]:
%run /content/drive/MyDrive/llm-perf-book-labs/setup/env_setup.py

## Step 4 — Log into Hugging Face

LLaMA-3.2 weights are gated. You need:
- A free account at [huggingface.co](https://huggingface.co)
- To have accepted Meta's license at [huggingface.co/meta-llama/Llama-3.2-1B](https://huggingface.co/meta-llama/Llama-3.2-1B)
- A read-access token from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)

**Recommended — store your token as a Colab secret (one-time):**
1. In Colab: click the 🔑 **Secrets** icon in the left sidebar
2. Add a new secret named `HF_TOKEN`, paste your token as the value, toggle **Notebook access** on
3. Re-run this cell — it will log in automatically every session

If you skip the secret, the cell falls back to a manual prompt.

In [ ]:
from huggingface_hub import login

try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'), add_to_git_credential=False)
    print("✓ Logged in via Colab secret HF_TOKEN")
except Exception:
    login()  # Falls back to manual token prompt

## Step 5 — Download LLaMA-3.2-1B weights to Drive

Downloads the model weights from HuggingFace directly to your Drive at
`MyDrive/llm-perf-book-labs/models/Llama-3.2-1B/`. The download happens over Colab's
datacenter connection — roughly **30 seconds for 2.5 GB**. You will never
need to download this again.

If the cell is interrupted, just re-run it — `snapshot_download` resumes
from where it left off.

In [ ]:
from huggingface_hub import snapshot_download

MODEL_ID   = "meta-llama/Llama-3.2-1B"
MODEL_DIR  = "/content/drive/MyDrive/llm-perf-book-labs/models/Llama-3.2-1B"

print(f"Downloading {MODEL_ID} → {MODEL_DIR}")
print("This takes ~30 seconds on Colab's network ...\n")

snapshot_download(
    repo_id=MODEL_ID,
    local_dir=MODEL_DIR,
    ignore_patterns=["*.bin"],   # prefer .safetensors; skip redundant .bin files
)

print("\n✓  Model weights saved to", MODEL_DIR)

---

## You're done

Your Drive now contains:

```
MyDrive/llm-perf-book-labs/
  setup/
    env_setup.py            ← setup script used by every chapter notebook
  notebooks/
    00_first_time_setup.ipynb
    chapter_00_*.ipynb
    chapter_01_*.ipynb
    ...                     ← all chapter notebooks, ready to open
  models/
    Llama-3.2-1B/           ← weights, persistent across sessions
```

**Close this notebook.** To start the book, open `chapter_00_*.ipynb` from
Colab via **File → Open notebook → Google Drive → MyDrive/llm-perf-book-labs/notebooks/**.

Each chapter notebook starts with the same three cells:
1. Mount Drive
2. `%run /content/drive/MyDrive/llm-perf-book-labs/setup/env_setup.py`
3. HF login (via `HF_TOKEN` secret, or manual prompt as fallback)

Run those three, see `✓  Environment ready.`, and you're on a working T4.